In [11]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader


In [13]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./MNIST",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./MNIST",
    train=False,
    download=True,
    transform=transform
)

In [14]:
trainloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=64)

In [15]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), #(inp, no. of filters,.,.)
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [16]:
model = CNN()

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [26]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.12431904671342373
epoch=2/10 & loss=0.1157712433010594
epoch=3/10 & loss=0.1127950087560019
epoch=4/10 & loss=0.11829993128131178
epoch=5/10 & loss=0.10857699998269187
epoch=6/10 & loss=0.10218706590804591
epoch=7/10 & loss=0.11403674667856015
epoch=8/10 & loss=0.11591409106871911
epoch=9/10 & loss=0.10266231661285165
epoch=10/10 & loss=0.10300057260011797


In [27]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 96.28


In [28]:
class RNN(nn.Module):

    def __init__(self, input_size=28, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):

        # x shape: (batch_size, 1, 28, 28)
        x = x.squeeze(1)

        # x shape: (batch_size, 28, 28)

        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        ).to(x.device)

        out, _ = self.rnn(x, h0)

        # Take output from the last time step
        out = out[:, -1, :]

        # Classification
        out = self.fc(out)

        return out


In [29]:
model = RNN()

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    epoch_training_loss = 0.0

    for images, labels in trainloader:

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        epoch_training_loss += loss.item()

    average_loss = epoch_training_loss / len(trainloader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 1/10, Loss: 0.8253
Epoch 2/10, Loss: 0.3352
Epoch 3/10, Loss: 0.2504


In [ ]:
model.eval()

correct_labels = 0
total_labels = 0

with torch.no_grad():

    for images, labels in testloader:

        outputs = model(images)

        # Get class with highest score
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()

        total_labels += labels.size(0)


accuracy = correct_labels / total_labels * 100

print(f"Accuracy: {accuracy:.2f}%")
